In [ ]:
import numpy as np
import math
import trimesh
import plotly.graph_objects as go
import Grasping_Face.grasping as gf
import Grasping_Face.visualize_grasping as gfv
import json

In [ ]:
import os
os.listdir('./models_target/models_cad/custom/')

In [ ]:
name = 'obj_000018' 

# target_mesh_file = f'./models_target/models_cad/custom/{name}.obj'
target_mesh_file = f'./models_target/models_cad/BOP_TLESS/{name}.obj'

In [ ]:
result = gf.compute_best_patch_pairs(
    mesh_path=target_mesh_file,
    mesh_max_triangles = 1000,
    angle_deg=10,              # 패치 병합 허용 각도 (↑면 패치 수 ↓)
    min_opening=1.0,           # 그리퍼 최소 개구(mm)
    max_opening=140.0,         # 그리퍼 최대 개구(mm)
    angle_tolerance_deg=15,    # 패치 페어 정반대 threshold
    top_k=500                  # 상위 패치 페어 후보 개수
)

reports = gf.check_gripper_feasibility_faces_with_yaw(result)
# reports = gf.check_gripper_feasibility_faces_with_rotation(result)

In [ ]:
fig = gfv.visualize_merged_patches_plotly(result, show=True)
# fig.write_html(f"./Grasping_Result/{name}_patch_normal.html", include_plotlyjs="cdn", full_html=True)

In [ ]:
fig = gfv.visualize_pairs_centroid_lines(result, show=True)
# fig.write_html(f"./Grasping_Result/{name}_patch_pairs.html", include_plotlyjs="cdn", full_html=True)

In [ ]:
fig = gfv.visualize_feasible_pairs(result, reports, show=True) # 선만 보기
# fig = gfv.visualize_feasible_pairs_with_cylinder(result, reports, show=True) # 실린더 보기
# fig.write_html(f"./Grasping_Result/{name}_patch_pairs_feasible.html", include_plotlyjs="cdn", full_html=True)

In [ ]:
# SAM6D result 
with open("./RUN_result/output/detection_pem_20251009_001130_object_100.json", "r") as f:
    detections = json.load(f)

# score 가장 높은 detection 선택
best_det = max(detections, key=lambda d: d["score"])
R_oc = np.array(best_det["R"])  # (3x3)
t_oc = np.array(best_det["t"]).reshape(3, 1)  # (3x1)

H_OC = gf.to44(R_oc, t_oc) # C좌표계 기준 O좌표계 원점 위치
H_OC[:3, :3], H_OC[:3, 3]

In [ ]:
from scipy.spatial.transform import Rotation

feasible_r = [r for r in reports if r.get('feasible')]
print(f'[GRP] feasible gripping sol.: {len(feasible_r)}')

mesh_patches = result['mesh_patches']
pad_params = gf.PadParams() # 패드 파라미터 미리 로드

feasible_pairs = []
H_dict = {"EE origin": np.eye(4,4)} # EE 좌표계 원점 
# for k in range(min(top_pairs, len(feasible_r))):
for k in range(len(feasible_r)):
    report = reports[k]
    pi = result['patches'][report['patch_i']]
    pj = result['patches'][report['patch_j']]
    ni = gf.unit(np.asarray(pi["normal"], float))
    nj = gf.unit(np.asarray(pj["normal"], float))
    fi = report['face_i']
    fj = report['face_j']
    ci = mesh_patches.vertices[mesh_patches.faces[fi]].mean(axis=0)
    cj = mesh_patches.vertices[mesh_patches.faces[fj]].mean(axis=0)
    yaw_deg = report['feasible_yaw']

    H_OG, stroke = gf.build_gripper_pose_obj_OPE({"centroid": ci, "normal": ni}, {"centroid": cj, "normal": nj}, yaw_deg, H_OC=H_OC)
    H_EdEn, H_OdEn = gf.ee_delta_pose_des(H_OC, H_OG)
    
    r_quat = Rotation.from_matrix(H_EdEn[:3,:3]).as_quat()

    # pair 검사 및 실제로 feasible한 pair 만 남김
    # 그리퍼 접근 각도 45 이하인 경우
    z_H_EdEn = H_EdEn[:3,2] # z축 각도 필터링용
    if np.dot(z_H_EdEn, np.array([0,0,1])) < 0.707: 
        continue
    # # 그리퍼 패드 - 바닥 간섭 검사
    pad_diagonal = np.sqrt(pad_params.pad_w**2 + pad_params.pad_h**2)
    pad_radius = pad_diagonal / 2.0
    H_OdEn_rot = H_OdEn[:3,:3]
    O_rotated = H_OdEn_rot @ mesh_patches.vertices.T
    z_check = O_rotated[-1,:].min() + pad_radius
    ci_world = H_OdEn_rot @ ci
    cj_world = H_OdEn_rot @ cj
    # print(f'{ci[2]:.3f}, {cj[2]:.3f}, {ci_world[2]:.3f}, {cj_world[2]:.3f}')
    if ci_world[2] < z_check or cj_world[2] < z_check:
        continue

    t_quat = H_EdEn[:3,3] / 1000
    res = {"pose_quat": np.concatenate([t_quat, r_quat]),
            "stroke": stroke}
    feasible_pairs.append(res)

    H_dict[f"pair {k+1}"] = H_EdEn


print(f'[GRP] feasible gripping sol.: {len(feasible_pairs)}')

In [ ]:
z_check

In [ ]:
gfv.visualize_frames(H_dict, scale=100, H_OdEn=H_OdEn, result=result)

In [ ]:
report = reports[1]
pi = result['patches'][report['patch_i']]
pj = result['patches'][report['patch_j']]
ni = gf.unit(np.asarray(pi["normal"], float))
nj = gf.unit(np.asarray(pj["normal"], float))
fi = report['face_i']
fj = report['face_j']
ci = result["mesh_patches"].vertices[result["mesh_patches"].faces[fi]].mean(axis=0)
cj = result["mesh_patches"].vertices[result["mesh_patches"].faces[fj]].mean(axis=0)
yaw_deg = report['feasible_yaw']

In [ ]:
yaw_deg

In [ ]:
H_OG, stroke = gf.build_gripper_pose_obj_OPE({"centroid": ci, "normal": ni}, {"centroid": cj, "normal": nj}, yaw_deg, H_OC)
H_OG[:3, :3], H_OG[:3, 3] # O좌표계 기준 G좌표계 원점 위치

In [ ]:
H_EdEn, H_OdEn = gf.ee_delta_pose_des(H_OC, H_OG)
H_EdEn[:3, :3], H_EdEn[:3, 3]

In [ ]:
H_OdEn[:3, :3], H_OdEn[:3, 3]

In [ ]:
from scipy.spatial.transform import Rotation
r_quat = Rotation.from_matrix(H_EdEn[:3,:3].T).as_quat()
t_quat = H_EdEn[:3,3] / 1000
r_quat

In [ ]:
from scipy.spatial.transform import Rotation

feasible_r = [r for r in reports if r.get('feasible')]
print(f'[GRP] feasible gripping sol.: {len(feasible_r)}')

mesh_patches = result['mesh_patches']

feasible_pairs = []
H_dict = {"EE origin": np.eye(4,4)} # EE 좌표계 원점 
for k in range(min(5, len(feasible_r))):
    report = reports[k]
    pi = result['patches'][report['patch_i']]
    pj = result['patches'][report['patch_j']]
    ni = gf.unit(np.asarray(pi["normal"], float))
    nj = gf.unit(np.asarray(pj["normal"], float))
    fi = report['face_i']
    fj = report['face_j']
    ci = mesh_patches.vertices[mesh_patches.faces[fi]].mean(axis=0)
    cj = mesh_patches.vertices[mesh_patches.faces[fj]].mean(axis=0)
    yaw_deg = report['feasible_yaw']

    H_OG, stroke = gf.build_gripper_pose_obj({"centroid": ci, "normal": ni}, {"centroid": cj, "normal": nj}, yaw_deg)
    H_EdEn, H_OdEn = gf.ee_delta_pose_des(H_OC, H_OG)
    
    r_quat = Rotation.from_matrix(H_EdEn[:3,:3]).as_quat()
    t_quat = H_EdEn[:3,3] / 1000
    res = {"pose_quat": np.concatenate([t_quat, r_quat]),
           "stroke": stroke}
    feasible_pairs.append(res)

    H_dict[f"pair {k+1}"] = H_EdEn

gfv.visualize_frames(H_dict, scale=100, H_OdEn=H_OdEn, result=result)

In [ ]:
# 좌표계 시각화
H_E = np.eye(4,4)

# 고정변환
H_GnEn = gf.to44(gf.Rotx(180) @ gf.Rotz(90), [0,0,-135])   # EE -> Grip  TODO: 로봇 컨트롤러 신호 받아 변환행렬 만들기
H_GnCn = gf.to44(np.eye(3), [0,48,6])           # Cam -> Grip
H_CnGn = np.linalg.inv(H_GnCn)
H_CnEn = H_GnEn @ H_CnGn                        # EE -> Cam
H_EG = np.linalg.inv(H_GnEn)

H_OdCn = H_OC
H_OdEn = H_CnEn @ H_OdCn
# H_GdOd = np.linalg.inv(H_OG)
H_GdEn = H_OdEn @ H_OG
H_EdEn = H_GdEn @ H_EG

H_dict = {
    "E": H_E, # 엔드이펙터
    "G": H_GnEn, 
    "C": H_CnEn,
    "O_d": H_OdEn, # 목표 
    "G_d": H_GdEn,
    "E_d": H_EdEn, # EE 상대 Pose
}
fig = gfv.visualize_frames(H_dict, scale=100)

In [ ]:
H_EdEn[:3, :3], H_EdEn[:3, 3]

In [1]:
import os
import numpy as np
import math
import trimesh
import plotly.graph_objects as go
import Grasping_Face.grasping as gf
import Grasping_Face.visualize_grasping as gfv

# for dataset_name in ['BOP_ITODD']: # 'custom' / 'BOP_TLESS' / 'BOP_IPD' / 'BOP_ITODD' / 'BOP_XYZIBD'
for dataset_name in ['BOP_XYZIBD', 'BOP_TLESS', 'BOP_IPD' , 'BOP_ITODD', 'custom']:

    dir_path = f'./Grasping_Result/{dataset_name}/'
    if not os.path.exists(dir_path):
        os.makedirs(dir_path)
    cads = [file for file in os.listdir(f'./models_target/models_cad/{dataset_name}') if file.endswith('.obj') or file.endswith('.ply')]

    for cad in cads:
        name = cad.split('.')[0]
        fmt  = cad.split('.')[-1]
        target_mesh_file = f'./models_target/models_cad/{dataset_name}/{name}.{fmt}'

        result = gf.compute_best_patch_pairs(
            mesh_path=target_mesh_file,
            mesh_max_triangles = 1500,
            angle_deg=15,              # 패치 병합 허용 각도 (↑면 패치 수 ↓)
            min_opening=1.0,           # 그리퍼 최소 개구(mm)
            max_opening=140.0,         # 그리퍼 최대 개구(mm)
            angle_tolerance_deg=15,    # 패치 페어 정반대 threshold
            top_k=500                  # 상위 패치 페어 후보 개수
        )

        reports = gf.check_gripper_feasibility_faces_with_yaw(result) # yaw (접근방향) 고려 O : 0, 90도 Pad Box 회전
        # reports = gf.check_gripper_feasibility_faces_with_rotation(result) # yaw (접근방향) 고려 X : 실린더


        feasible_p = result['top_k']
        feasible_r = [r for r in reports if r.get('feasible')]
        print(f'{dataset_name} - {name} : patch pairs {len(feasible_p)}, feasible sol. {len(feasible_r)}')

        fig = gfv.visualize_merged_patches_plotly(result)
        fig.write_html(f"./{dir_path}/3-patch_normal_{name}.html", include_plotlyjs="cdn", full_html=True)

        try:
            fig = gfv.visualize_pairs_centroid_lines(result)
            fig.write_html(f"./{dir_path}/2-patch_pairs_{name}.html", include_plotlyjs="cdn", full_html=True)
        except:
            print(f'cannot get patch pairs in {dataset_name}:{name}')
            pass

        try:
            # fig = gfv.visualize_feasible_pairs(result, reports)
            # fig = gfv.visualize_feasible_pairs_pads(result, reports) # yaw (접근방향) 고려 O : 0, 90도 Pad Box 회전, 패드 그리기
            fig = gfv.visualize_feasible_pairs_with_cylinder(result, reports) # yaw (접근방향) 고려 X : 실린더
            fig.write_html(f"./{dir_path}/1-grasping_pairs_feasible_{name}_{len(feasible_r)}sol.html", include_plotlyjs="cdn", full_html=True)
        except:
            print(f'no feasible pairs in {dataset_name}:{name}')
            pass

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
BOP_XYZIBD - obj_000001 : patch pairs 67, feasible sol. 838
BOP_XYZIBD - obj_000002 : patch pairs 46, feasible sol. 291
BOP_XYZIBD - obj_000004 : patch pairs 99, feasible sol. 265
BOP_XYZIBD - obj_000005 : patch pairs 102, feasible sol. 256
BOP_XYZIBD - obj_000006 : patch pairs 149, feasible sol. 546
BOP_XYZIBD - obj_000008 : patch pairs 98, feasible sol. 174
BOP_XYZIBD - obj_000009 : patch pairs 135, feasible sol. 3063
BOP_XYZIBD - obj_000010 : patch pairs 127, feasible sol. 831
BOP_XYZIBD - obj_000011 : patch pairs 66, feasible sol. 400
BOP_XYZIBD - obj_000012 : patch pairs 15, feasible sol. 3260
BOP_XYZIBD - obj_000013 : patch pairs 36, feasible sol. 543
BOP_XYZIBD - obj_000014 : patch pairs 26, feasible sol. 280
BOP_XYZIBD - obj_000015 : patch pairs 149, feasible sol. 147
BOP_XYZIBD - obj_000016 : patch pairs 105, f

d:\PythonDev\OPE_Grasping\Grasping_Face\grasping.py:593: RuntimeWarning:

invalid value encountered in divide



BOP_ITODD - obj_000014 : patch pairs 180, feasible sol. 196
BOP_ITODD - obj_000015 : patch pairs 500, feasible sol. 275
BOP_ITODD - obj_000016 : patch pairs 93, feasible sol. 326
BOP_ITODD - obj_000017 : patch pairs 500, feasible sol. 243
BOP_ITODD - obj_000018 : patch pairs 154, feasible sol. 451
BOP_ITODD - obj_000019 : patch pairs 3, feasible sol. 602
BOP_ITODD - obj_000020 : patch pairs 272, feasible sol. 129
BOP_ITODD - obj_000021 : patch pairs 65, feasible sol. 1065
BOP_ITODD - obj_000022 : patch pairs 428, feasible sol. 161
BOP_ITODD - obj_000023 : patch pairs 217, feasible sol. 1070
BOP_ITODD - obj_000024 : patch pairs 47, feasible sol. 846
BOP_ITODD - obj_000025 : patch pairs 64, feasible sol. 0
no feasible pairs in BOP_ITODD:obj_000025
BOP_ITODD - obj_000026 : patch pairs 267, feasible sol. 416
BOP_ITODD - obj_000027 : patch pairs 107, feasible sol. 188
BOP_ITODD - obj_000028 : patch pairs 27, feasible sol. 104
custom - bracket_1 : patch pairs 144, feasible sol. 144
custom - 

#### Test

In [ ]:
import numpy as np
import math
import trimesh
import plotly.graph_objects as go
import Grasping_Face.grasping as gf
import Grasping_Face.visualize_grasping as gfv
import json

In [ ]:
name = 'obj_000018' # logitech_c930e_m / obj_000001
target_mesh_file = f'./models_target/models_cad/BOP_TLESS/{name}.obj'

In [ ]:
mesh_path=target_mesh_file
mesh_max_triangles = 1000
angle_deg=15              # 패치 병합 허용 각도 (↑면 패치 수 ↓)
min_opening=1.0           # 그리퍼 최소 개구(mm)
max_opening=140.0         # 그리퍼 최대 개구(mm)
angle_tolerance_deg=15    # 패치 페어 정반대 threshold
top_k=500                  # 상위 패치 페어 후보 개수

In [ ]:
result['top_k'][24]

In [ ]:
mesh_filter, mesh_quad = gf.load_uniform_mesh_with_open3d(mesh_path, target_triangles=mesh_max_triangles)

mesh_COM = mesh_quad.center_mass
mesh_bound = float(np.linalg.norm(mesh_quad.bounds[1] - mesh_quad.bounds[0]))
# mesh_filter: (면적 기준 필터링 이후, watertight X), mesh_quad: (면적 기준 필터링 이전, watertight O)
remesh = gf.split_long_edges(mesh_filter)
# remesh = gf.merge_small_faces(remesh)

patches = gf.extract_planar_patches(remesh, angle_deg=angle_deg) # coplanar_tol 삭제
patches = gf.orient_patch_normals(mesh_quad, remesh, patches)  # 모두 바깥쪽으로 정렬

params = gf.PatchPairParams(
    min_opening=min_opening,
    max_opening=max_opening,
    angle_tolerance_deg=angle_tolerance_deg,
)

In [ ]:
cand = result['top_k'][24]
mesh = result['mesh_patches']
mesh_ch = result['mesh_quad']
patches = result['patches']
com = mesh_ch.center_mass

clearance_out = 10.0
pad_w = 34
pad_h = 21
pad_d = 7

cm = trimesh.collision.CollisionManager()
cm.add_object("part", mesh_ch) # 간섭 검사용: 면적 필터 안 된 메시 

In [ ]:
pid_i, pid_j = cand["patch_i"], cand["patch_j"]

p_i, p_j = patches[pid_i], patches[pid_j]
n_i = gf.unit(np.asarray(p_i["normal"], float))
n_j = gf.unit(np.asarray(p_j["normal"], float))

# pair별로 가능한 모든 후보를 찾기
reports_for_this_pair = []
mesh_F, mesh_V = mesh.faces, mesh.vertices
for f_i in p_i["face_indices"]:
    ci = mesh_V[mesh_F[f_i]].mean(axis=0)

    best_j   = None
    best_cj  = None
    best_scr = -1.0
    for f_j in p_j["face_indices"]:
        cj = mesh_V[mesh_F[f_j]].mean(axis=0)
        d  = cj - ci
        nd = np.linalg.norm(d)
        if nd < 1e-12:
            continue
        d_hat = d / nd
        scr = abs(float(d_hat @ (-n_i))) * abs(float((-d_hat) @ n_j))
        if scr > best_scr:
            best_scr, best_j, best_cj = scr, f_j, cj
    if best_j is None:
        continue
    # 엇갈린 face pair 건너뛰기
    alignment_dist_i = gf.point_line_distance(ci, -n_i, best_cj)
    alignment_dist_j = gf.point_line_distance(best_cj, -n_j, ci)
    # dist_criteria = min(pad_w, pad_h) / 5 # 엇갈림 기준
    dist_criteria = 10 # 엇갈림 기준
    if alignment_dist_i > dist_criteria or alignment_dist_j > dist_criteria:
        print(f'alignment error, {f_i} {f_j} ')


    yaw=90
    box_i = gf.make_rot_pad_box_at_patch(
        {"centroid": ci, "normal": n_i},
        pad_w, pad_h, pad_d, clearance_out, yaw_deg=yaw
    )
    box_j = gf.make_rot_pad_box_at_patch(
        {"centroid": best_cj, "normal": n_j},
        pad_w, pad_h, pad_d, clearance_out, yaw_deg=-yaw
    )
    # 충돌이 발생하면 이 yaw는 건너뛰고 다음 yaw를 검사.
    if cm.in_collision_single(box_i): print(f'collision, {f_i} {f_j}')
    if cm.in_collision_single(box_j): print(f'collision, {f_i} {f_j}')

                    
    midpoint = 0.5 * (ci + best_cj)
    current_dist = np.linalg.norm(midpoint - com)
    
    Fi = -n_i; Fj = -n_j
    tau = np.cross(ci - com, Fi) + np.cross(best_cj - com, Fj)
    current_moment = float(np.linalg.norm(tau))

In [ ]:
tau